**Linear Regression_MLR**

In [62]:
#Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

In [63]:
# Importing the dataset
dataset = pd.read_csv('C:/MAFAS/APU/CT046-3-M-AML/CT046 - LABS/Python LAB MATERIALS/Lab 7 - Cross Validation/kc_house_data.csv')
dataset.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [64]:
dataset.isnull().sum()

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

In [65]:
x = dataset[dataset.columns[3:21]] # IV
y = dataset[dataset.columns[2]] # price

In [66]:
x.columns

Index(['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
       'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [67]:
len(x.columns)

18

In [68]:
y.head()

0    221900.0
1    538000.0
2    180000.0
3    604000.0
4    510000.0
Name: price, dtype: float64

In [69]:
X = sm.add_constant(x)
est = sm.OLS(y, X).fit()
print(est.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.700
Model:                            OLS   Adj. R-squared:                  0.700
Method:                 Least Squares   F-statistic:                     2960.
Date:                Thu, 08 Jan 2026   Prob (F-statistic):               0.00
Time:                        22:28:39   Log-Likelihood:            -2.9460e+05
No. Observations:               21613   AIC:                         5.892e+05
Df Residuals:                   21595   BIC:                         5.894e+05
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           6.69e+06   2.93e+06      2.282

In [70]:
# Linear Regression
from sklearn.linear_model import LinearRegression
reg = LinearRegression()
reg.get_params()

{'copy_X': True,
 'fit_intercept': True,
 'n_jobs': None,
 'positive': False,
 'tol': 1e-06}

In [71]:
# predicting on X_test....This is a rare occasion to predict the TV using one IV
reg.fit(x, y)
y_pred = reg.predict(x)
y_pred.round(2)

array([208877.95, 734051.82, 380504.72, ..., 143431.95, 385287.12,
       146084.83], shape=(21613,))

In [72]:
pd.DataFrame({'Actual': y.round(2), 'Predicted': y_pred.round(2)})[0:5]

,Actual,Predicted
0,221900.0,208877.95
1,538000.0,734051.82
2,180000.0,380504.72
3,604000.0,455050.87
4,510000.0,440955.63


In [73]:
# Return the R^2 
round(reg.score(x,y), 2)

0.7

In [74]:
# calculating MSE, RMSE, MAE
from sklearn.metrics import mean_squared_error, mean_absolute_error
print("Mean Squared Error: ", round(mean_squared_error(y, y_pred), 2))
print('Mean Absolute Error: ', round(mean_absolute_error(y, y_pred), 2))  
print('Root Mean Squared Error: ', round(np.sqrt(mean_squared_error(y, y_pred)), 2))

Mean Squared Error:  40466915557.49
Mean Absolute Error:  125922.65
Root Mean Squared Error:  201163.9


**Cross Validation**

Learning the parameters of a prediction function and testing it on the same data is a methodological mistake: a model that would just repeat the labels of the samples that it has just seen would have a perfect score but would fail to predict anything useful on yet-unseen data. This situation is called **overfitting**. To avoid it, it is common practice when performing a (supervised) machine learning experiment to hold out part of the available data as a test set **x_test, y_test**.

To solve this problem, yet another part of the dataset can be held out as a so-called **validation set**: training proceeds on the training set, after which evaluation is done on the validation set, and when the experiment seems to be successful, final evaluation can be done on the test set.

However, by partitioning the available data into three sets, we drastically reduce the number of samples which can be used for learning the model, and the results can depend on a particular random choice for the pair of (train, validation) sets.

A solution to this problem is a procedure called **cross-validation (CV)**. A test set should still be held out for final evaluation, but the validation set is no longer needed when doing CV.

**Scoring Parameter**

See The scoring parameter: defining model evaluation rules for details. 

https://scikit-learn.org/stable/modules/model_evaluation.html

**R^2**

In [75]:
from sklearn.model_selection import cross_val_score
R2s = cross_val_score(reg, x, y, cv = 5, scoring = 'r2')
print(R2s)
print("Average R Square after CV: ", np.mean(R2s))

[0.69615715 0.69103231 0.69273185 0.70769239 0.68534044]
Average R Square after CV:  0.69459082832835


**MSE**

In [76]:
MSEs = cross_val_score(reg, x, y, cv = 5, scoring = 'neg_mean_squared_error')
print(MSEs)
print("Average MSE after CV: ", np.mean(MSEs).round(2))

[-4.60248841e+10 -4.33255121e+10 -3.58941035e+10 -3.71043942e+10
 -4.32225225e+10]
Average MSE after CV:  -41114283280.91


**K-Fold CV**

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html#sklearn.model_selection.KFold

The performance measure reported by k-fold cross-validation is then the average of the values computed in the loop. This approach can be computationally expensive, but does not waste too much data (as is the case when fixing an arbitrary validation set), which is a major advantage in problems such as inverse inference where the number of samples is very small.

In [77]:
from sklearn.model_selection import KFold
folds = KFold(n_splits = 5, shuffle = True, random_state = 100)

R2s_kcv = cross_val_score(reg, x, y, cv = folds, scoring = 'r2')
print(R2s_kcv)
print("Average R Square after K-Fold CV: ", np.mean(R2s_kcv))

[0.70290889 0.66876172 0.69829753 0.72239843 0.69646199]
Average R Square after K-Fold CV:  0.6977657131176525


**Shuffle Split**

It is also possible to use other cross validation strategies by passing a cross validation iterator

In [78]:
from sklearn.model_selection import ShuffleSplit
Shuffle_Split = ShuffleSplit(n_splits = 5, test_size = 0.2, random_state = 42)

R2s_sscv = cross_val_score(reg, x, y, cv = Shuffle_Split, scoring = 'r2')
print(R2s_sscv)
print("Average R Square after K-Fold CV: ", np.mean(R2s_sscv))

[0.70119044 0.69594072 0.70265633 0.71209084 0.68575801]
Average R Square after K-Fold CV:  0.6995272690050441


**Stratified KFold**

Stratified K-Folds cross-validator.

Provides train/test indices to split data in train/test sets.

This cross-validation object is a variation of KFold that returns stratified folds. The folds are made by preserving the percentage of samples for each class.

In [79]:
from sklearn.model_selection import StratifiedKFold

skfolds = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42, )

R2s_skcv = cross_val_score(reg, x, y, cv = skfolds, scoring = 'r2')
print(R2s_skcv)
print("Average R Square after Stratified K-Fold CV: ", np.mean(R2s_skcv))

[0.69497786 0.70113664 0.69360683 0.70270108 0.69673076]
Average R Square after Stratified K-Fold CV:  0.6978306330393698


**Repeated KFold**

In [80]:
from sklearn.model_selection import RepeatedKFold
rkfolds = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = 42)

R2s_rkcv = cross_val_score(reg, x, y, cv = rkfolds, scoring = 'r2')
print(R2s_rkcv)
print("Average R Square after Repeated K-Fold CV: ", np.mean(R2s_rkcv))

[0.70119044 0.68597027 0.69547324 0.70465746 0.7062373  0.69594072
 0.700825   0.70285474 0.69048405 0.69423036 0.70265633 0.68917779
 0.70337874 0.69619247 0.6979725  0.71209084 0.68947412 0.68097454
 0.70779081 0.70510383 0.68575801 0.70104533 0.70253705 0.70448614
 0.69783716]
Average R Square after Repeated K-Fold CV:  0.6981735708977923


**Comparing**

In [81]:
print("Average R Square after CV: ", np.mean(R2s))
print("Average R Square after K-Fold CV: ", np.mean(R2s_kcv))
print("Average R Square after Stratified K-Fold CV: ", np.mean(R2s_skcv))
print("Average R Square after Repeated K-Fold CV: ", np.mean(R2s_rkcv))

Average R Square after CV:  0.69459082832835
Average R Square after K-Fold CV:  0.6977657131176525
Average R Square after Stratified K-Fold CV:  0.6978306330393698
Average R Square after Repeated K-Fold CV:  0.6981735708977923


In [82]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV, RepeatedKFold

# step-1: create a cross-validation scheme
rkfolds = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = 42)

# step-2: specify range of hyperparameters to tune
hyper_params = [{'n_features_to_select': list(range(1, 14))}]


# step-3: perform grid search
# 3.1 specify model
lm = LinearRegression()
lm.fit(x, y)
rfe = RFE(lm, step = 1)             

# 3.2 call GridSearchCV()
model_rkf_cv = GridSearchCV(estimator = rfe, param_grid = hyper_params, scoring = 'r2', cv = rkfolds, verbose = 1, return_train_score = True)      

# fit the model
model_rkf_cv.fit(x, y)   

Fitting 25 folds for each of 13 candidates, totalling 325 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RFE(estimator...rRegression())
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'n_features_to_select': [1, 2, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",RepeatedKFold...ndom_state=42)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also disp

In [83]:
y_pred_2 = model_rkf_cv.predict(x)
print("Best Parameter: ", model_rkf_cv.best_params_)
print("Best R-Squared Value: ", model_rkf_cv.best_score_)

Best Parameter:  {'n_features_to_select': 13}
Best R-Squared Value:  0.6972116188828578


In [84]:
pd.DataFrame({'Actual': y.round(2), 'Predicted': y_pred_2.round(2)})[0:5]

,Actual,Predicted
0,221900.0,212509.96
1,538000.0,704053.21
2,180000.0,354713.05
3,604000.0,458904.46
4,510000.0,444726.04
